## Импорты

In [1]:
import pandas as pd
import numpy as np
import itertools
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, PredefinedSplit
from sklearn.metrics import root_mean_squared_error, accuracy_score, classification_report, make_scorer, fbeta_score
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestRegressor, BaggingRegressor, VotingRegressor, StackingRegressor,
    RandomForestClassifier, VotingClassifier, StackingClassifier
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import time
import requests
import asyncio
import random
import re
from urllib.parse import quote
from playwright.async_api import async_playwright

In [2]:
warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

## Подготовка данных

In [3]:
df = pd.read_csv('data/epi_r.csv')
df.head(1)

,title,rating,calories,protein,fat,sodium,#cakeweek,#wasteless,22-minute meals,3-ingredient recipes,30 days of groceries,advance prep required,alabama,alaska,alcoholic,almond,amaretto,anchovy,anise,anniversary,anthony bourdain,aperitif,appetizer,apple,apple juice,apricot,arizona,artichoke,arugula,asian pear,asparagus,aspen,atlanta,australia,avocado,back to school,backyard bbq,bacon,bake,banana,barley,basil,bass,bastille day,bean,beef,beef rib,beef shank,beef tenderloin,beer,beet,bell pepper,berry,beverly hills,birthday,biscuit,bitters,blackberry,blender,blue cheese,blueberry,boil,bok choy,bon appétit,bon app��tit,boston,bourbon,braise,bran,brandy,bread,breadcrumbs,breakfast,brie,brine,brisket,broccoli,broccoli rabe,broil,brooklyn,brown rice,brownie,brunch,brussel sprout,buffalo,buffet,bulgaria,bulgur,burrito,butter,buttermilk,butternut squash,butterscotch/caramel,cabbage,cake,california,calvados,cambridge,campari,camping,canada,candy,candy thermometer,cantaloupe,capers,caraway,cardamom,carrot,cashew,casserole/gratin,cauliflower,caviar,celery,chambord,champagne,chard,chartreuse,cheddar,cheese,cherry,chestnut,chicago,chicken,chickpea,chile,chile pepper,chili,chill,chive,chocolate,christmas,christmas eve,cilantro,cinco de mayo,cinnamon,citrus,clam,clove,cobbler/crumble,cocktail,cocktail party,coconut,cod,coffee,coffee grinder,cognac/armagnac,collard greens,colorado,columbus,condiment,condiment/spread,connecticut,cook like a diner,cookbook critic,cookie,cookies,coriander,corn,cornmeal,costa mesa,cottage cheese,couscous,crab,cranberry,cranberry sauce,cream cheese,créme de cacao,crêpe,cr��me de cacao,cuba,cucumber,cumin,cupcake,currant,curry,custard,dairy,dairy free,dallas,date,deep-fry,denver,dessert,digestif,dill,dinner,dip,diwali,dominican republic,dorie greenspan,double boiler,dried fruit,drink,drinks,duck,easter,eau de vie,edible gift,egg,egg nog,eggplant,egypt,emeril lagasse,endive,engagement party,england,entertaining,epi + ushg,epi loves the microwave,escarole,fall,family reunion,fat free,father's day,fennel,feta,fig,fish,flaming hot summer,flat bread,florida,fontina,food processor,fortified wine,fourth of july,france,frangelico,frankenrecipe,freeze/chill,freezer food,friendsgiving,frittata,fritter,frozen dessert,fruit,fruit juice,fry,game,garlic,georgia,germany,gin,ginger,goat cheese,goose,gouda,gourmet,graduation,grains,grand marnier,granola,grape,grapefruit,grappa,green bean,green onion/scallion,grill,grill/barbecue,ground beef,ground lamb,guam,guava,haiti,halibut,halloween,ham,hamburger,hanukkah,harpercollins,hawaii,hazelnut,healdsburg,healthy,herb,high fiber,hollywood,hominy/cornmeal/masa,honey,honeydew,hors d'oeuvre,horseradish,hot drink,hot pepper,house & garden,house cocktail,houston,hummus,ice cream,ice cream machine,iced coffee,iced tea,idaho,illinois,indiana,iowa,ireland,israel,italy,jalapeño,jam or jelly,jamaica,japan,jerusalem artichoke,juicer,jícama,kahlúa,kale,kansas,kansas city,kentucky,kentucky derby,kid-friendly,kidney friendly,kirsch,kitchen olympics,kiwi,kosher,kosher for passover,kumquat,kwanzaa,labor day,lamb,lamb chop,lamb shank,lancaster,las vegas,lasagna,leafy green,leek,legume,lemon,lemon juice,lemongrass,lentil,lettuce,lima bean,lime,lime juice,lingonberry,liqueur,lobster,london,long beach,los angeles,louisiana,louisville,low cal,low carb,low cholesterol,low fat,low sodium,low sugar,low/no sugar,lunar new year,lunch,lychee,macadamia nut,macaroni and cheese,maine,mandoline,mango,maple syrup,mardi gras,margarita,marinade,marinate,marsala,marscarpone,marshmallow,martini,maryland,massachusetts,mayonnaise,meat,meatball,meatloaf,melon,mexico,mezcal,miami,michigan,microwave,midori,milk/cream,minneapolis,minnesota,mint,mississippi,missouri,mixer,molasses,monterey jack,mortar and pestle,mother's day,mozzarella,muffin,mushroom,mussel,mustard,mustard greens,nancy silverton,nebraska,nectarine,new hampshire,new jersey,new mexico,new orleans,new year's day,new year's eve,new york,"no meat, no problem",no sugar a

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20052 entries, 0 to 20051
Columns: 680 entries, title to turkey
dtypes: float64(679), str(1)
memory usage: 104.0 MB


Очистим данные от дубликатов и от строк с NaN-значениями

In [5]:
df = df.dropna()
df = df.drop_duplicates(keep='last')
df = df.drop_duplicates(subset='title', keep='last')

print(f'Количество NaN: {df.isnull().sum().sum()}')
print(f'Количество дубликатов: {df.duplicated().sum()}')

Количество NaN: 0
Количество дубликатов: 0


In [6]:
df.to_csv('data/clean_dataset.csv', index=False)

Признак, который мы предсказываем - рейтинг рецепта

In [7]:
X = df.drop(columns=['rating'])
y = df['rating']

С помощью LLM был отобран список не-ингредиентов, который будет удалён из X

In [8]:
EXCLUDE_COLUMNS = [
    '#cakeweek',
    '#wasteless',
    '22-minute meals',
    '3-ingredient recipes',
    '30 days of groceries',
    'advance prep required',
    'alabama',
    'alaska',
    'alcoholic',
    'anniversary',
    'anthony bourdain',
    'aperitif',
    'appetizer',
    'arizona',
    'aspen',
    'atlanta',
    'australia',
    'back to school',
    'backyard bbq',
    'bake',
    'bastille day',
    'beverly hills',
    'birthday',
    'biscuit',
    'blender',
    'boil',
    'bon app��tit',
    'bon appétit',
    'boston',
    'braise',
    'bread',
    'breakfast',
    'brine',
    'broil',
    'brooklyn',
    'brownie',
    'brunch',
    'buffet',
    'bulgaria',
    'burrito',
    'cake',
    'california',
    'calories',
    'cambridge',
    'camping',
    'canada',
    'candy',
    'candy thermometer',
    'casserole/gratin',
    'chicago',
    'chill',
    'christmas',
    'christmas eve',
    'cinco de mayo',
    'cobbler/crumble',
    'cocktail',
    'cocktail party',
    'coffee grinder',
    'colorado',
    'columbus',
    'condiment',
    'condiment/spread',
    'connecticut',
    'cook like a diner',
    'cookbook critic',
    'cookbooks',
    'cookie',
    'cookies',
    'costa mesa',
    'crêpe',
    'cuba',
    'cupcake',
    'custard',
    'dairy',
    'dairy free',
    'dallas',
    'deep-fry',
    'denver',
    'dessert',
    'digestif',
    'dinner',
    'dip',
    'diwali',
    'dominican republic',
    'dorie greenspan',
    'double boiler',
    'drink',
    'drinks',
    'easter',
    'edible gift',
    'egypt',
    'emeril lagasse',
    'engagement party',
    'england',
    'entertaining',
    'epi + ushg',
    'epi loves the microwave',
    'fall',
    'family reunion',
    'fat',
    'fat free',
    "father's day",
    'flaming hot summer',
    'flat bread',
    'florida',
    'food processor',
    'fortified wine',
    'fourth of july',
    'france',
    'frankenrecipe',
    'freeze/chill',
    'freezer food',
    'friendsgiving',
    'frittata',
    'fritter',
    'frozen dessert',
    'fry',
    'georgia',
    'germany',
    'gourmet',
    'graduation',
    'grains',
    'granola',
    'grill',
    'grill/barbecue',
    'guam',
    'haiti',
    'halloween',
    'hamburger',
    'hanukkah',
    'harpercollins',
    'hawaii',
    'healdsburg',
    'healthy',
    'high fiber',
    'hollywood',
    "hors d'oeuvre",
    'hot drink',
    'house & garden',
    'house cocktail',
    'houston',
    'ice cream',
    'ice cream machine',
    'iced coffee',
    'iced tea',
    'idaho',
    'illinois',
    'indiana',
    'iowa',
    'ireland',
    'israel',
    'italy',
    'jamaica',
    'japan',
    'juicer',
    'kansas',
    'kansas city',
    'kentucky',
    'kentucky derby',
    'kid-friendly',
    'kidney friendly',
    'kitchen olympics',
    'kosher',
    'kosher for passover',
    'kwanzaa',
    'labor day',
    'lancaster',
    'las vegas',
    'lasagna',
    'leftovers',
    'liqueur',
    'london',
    'long beach',
    'los angeles',
    'louisiana',
    'louisville',
    'low cal',
    'low carb',
    'low cholesterol',
    'low fat',
    'low sodium',
    'low sugar',
    'low/no sugar',
    'lunar new year',
    'lunch',
    'macaroni and cheese',
    'maine',
    'mandoline',
    'mardi gras',
    'marinade',
    'marinate',
    'martini',
    'maryland',
    'massachusetts',
    'meatball',
    'meatloaf',
    'mexico',
    'miami',
    'michigan',
    'microwave',
    'minneapolis',
    'minnesota',
    'mississippi',
    'missouri',
    'mixer',
    'mortar and pestle',
    "mother's day",
    'muffin',
    'nancy silverton',
    'nebraska',
    'new hampshire',
    'new jersey',
    'new mexico',
    'new orleans',
    "new year's day",
    "new year's eve",
    'new york',
    'no meat, no problem',
    'no sugar added',
    'no-cook',
    'non-alcoholic',
    'north carolina',
    'ohio',
    'oklahoma',
    'oktoberfest',
    'omelet',
    'one-pot meal',
    'oregon',
    'organic',
    'oscars',
    'pacific palisades',
    'paleo',
    'pan-fry',
    'pancake',
    'parade',
    'paris',
    'party',
    'pasadena',
    'passover',
    'pasta maker',
    'peanut free',
    'pennsylvania',
    'persian new year',
    'peru',
    'pescatarian',
    'philippines',
    'picnic',
    'pie',
    'pittsburgh',
    'pizza',
    'poach',
    'poker/game night',
    'portland',
    'pot pie',
    'potato salad',
    'potluck',
    'pressure cooker',
    'protein',
    'providence',
    'punch',
    'purim',
    'quiche',
    'quick & easy',
    'quick and healthy',
    'ramadan',
    'ramekin',
    'raw',
    'rhode island',
    'roast',
    'rosh hashanah/yom kippur',
    'rub',
    'salad',
    'salad dressing',
    'san francisco',
    'sandwich',
    'sandwich theory',
    'sangria',
    'santa monica',
    'sauce',
    'sauté',
    'seattle',
    'self',
    'shavuot',
    'shower',
    'side',
    'simmer',
    'skewer',
    'slow cooker',
    'smoker',
    'smoothie',
    'snack',
    'snack week',
    'sodium',
    'sorbet',
    'soufflé/meringue',
    'soup/stew',
    'south carolina',
    'soy free',
    'spain',
    'spirit',
    'spring',
    'spritzer',
    'st. louis',
    "st. patrick's day",
    'steam',
    'stew',
    'stir-fry',
    'stuffing/dressing',
    'sugar conscious',
    'sukkot',
    'summer',
    'super bowl',
    'suzanne goin',
    'switzerland',
    'taco',
    'tailgating',
    'tart',
    'tennessee',
    'tested & improved',
    'texas',
    'thanksgiving',
    'title',
    'tree nut free',
    'utah',
    "valentine's day",
    'vegan',
    'vegetarian',
    'vermont',
    'virginia',
    'waffle',
    'washington',
    'washington, d.c.',
    'wedding',
    'weelicious',
    'west virginia',
    'westwood',
    'wheat/gluten-free',
    'windsor',
    'winter',
    'wisconsin',
    'wok',
    'yonkers',
]

In [9]:
X = X.drop(columns=EXCLUDE_COLUMNS)
X.head(1)

,almond,amaretto,anchovy,anise,apple,apple juice,apricot,artichoke,arugula,asian pear,asparagus,avocado,bacon,banana,barley,basil,bass,bean,beef,beef rib,beef shank,beef tenderloin,beer,beet,bell pepper,berry,bitters,blackberry,blue cheese,blueberry,bok choy,bourbon,bran,brandy,breadcrumbs,brie,brisket,broccoli,broccoli rabe,brown rice,brussel sprout,buffalo,bulgur,butter,buttermilk,butternut squash,butterscotch/caramel,cabbage,calvados,campari,cantaloupe,capers,caraway,cardamom,carrot,cashew,cauliflower,caviar,celery,chambord,champagne,chard,chartreuse,cheddar,cheese,cherry,chestnut,chicken,chickpea,chile,chile pepper,chili,chive,chocolate,cilantro,cinnamon,citrus,clam,clove,coconut,cod,coffee,cognac/armagnac,collard greens,coriander,corn,cornmeal,cottage cheese,couscous,crab,cranberry,cranberry sauce,cream cheese,créme de cacao,cr��me de cacao,cucumber,cumin,currant,curry,date,dill,dried fruit,duck,eau de vie,egg,egg nog,eggplant,endive,escarole,fennel,feta,fig,fish,fontina,frangelico,fruit,fruit juice,game,garlic,gin,ginger,goat cheese,goose,gouda,grand marnier,grape,grapefruit,grappa,green bean,green onion/scallion,ground beef,ground lamb,guava,halibut,ham,hazelnut,herb,hominy/cornmeal/masa,honey,honeydew,horseradish,hot pepper,hummus,jalapeño,jam or jelly,jerusalem artichoke,jícama,kahlúa,kale,kirsch,kiwi,kumquat,lamb,lamb chop,lamb shank,leafy green,leek,legume,lemon,lemon juice,lemongrass,lentil,lettuce,lima bean,lime,lime juice,lingonberry,lobster,lychee,macadamia nut,mango,maple syrup,margarita,marsala,marscarpone,marshmallow,mayonnaise,meat,melon,mezcal,midori,milk/cream,mint,molasses,monterey jack,mozzarella,mushroom,mussel,mustard,mustard greens,nectarine,noodle,nut,nutmeg,oat,oatmeal,octopus,okra,olive,onion,orange,orange juice,oregano,orzo,oyster,papaya,paprika,parmesan,parsley,parsnip,passion fruit,pasta,pastry,pea,peach,peanut,peanut butter,pear,pecan,pepper,pernod,persimmon,phyllo/puff pastry dough,pickles,pine nut,pineapple,pistachio,plantain,plum,poblano,pomegranate,pomegranate juice,poppy,pork,pork chop,pork rib,pork tenderloin,port,potato,poultry,poultry sausage,prosciutto,prune,pumpkin,quail,quince,quinoa,rabbit,rack of lamb,radicchio,radish,raisin,raspberry,red wine,rhubarb,rice,ricotta,root vegetable,rosemary,rosé,rum,rutabaga,rye,saffron,sage,sake,salmon,salsa,sardine,sausage,scallop,scotch,seafood,seed,semolina,sesame,sesame oil,shallot,shellfish,sherry,shrimp,snapper,sour cream,sourdough,soy,soy sauce,sparkling wine,spice,spinach,squash,squid,steak,stock,strawberry,sugar snap pea,sweet potato/yam,swiss cheese,swordfish,tamarind,tangerine,tapioca,tarragon,tea,tequila,thyme,tilapia,tofu,tomatillo,tomato,tortillas,tree nut,triple sec,tropical fruit,trout,tuna,turnip,vanilla,veal,vegetable,venison,vermouth,vinegar,vodka,walnut,wasabi,watercress,watermelon,whiskey,white wine,whole wheat,wild rice,wine,yellow squash,yogurt,yuca,zucchini,turkey
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0

In [10]:
X.info()

<class 'pandas.DataFrame'>
Index: 14058 entries, 0 to 20051
Columns: 337 entries, almond to turkey
dtypes: float64(337)
memory usage: 36.3 MB


Итого осталось 337 признаков. Поскольку они все закодированы с помощью OHE, стандартизация данных не нужна

Разбиение на обучающую/валидационную/тестовую выборку будет в соотношении 72/8/20

In [11]:
RANDOM_STATE = 21

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_val, y_train_val, test_size=0.1, random_state=RANDOM_STATE)

print(f'Обучающая выборка: {X_train.shape, y_train.shape}')
print(f'Валидационная выборка: {X_valid.shape, y_valid.shape}')
print(f'Тестовая выборка: {X_test.shape, y_test.shape}')

Обучающая выборка: ((10121, 337), (10121,))
Валидационная выборка: ((1125, 337), (1125,))
Тестовая выборка: ((2812, 337), (2812,))


## Регрессия

### 1. Наивная регрессия

In [12]:
naive_mean = y_train.mean()
naive_pred = np.full(len(y_test), naive_mean)
naive_rmse = root_mean_squared_error(y_test, naive_pred)

print(f"Средний рейтинг в train: {naive_mean:.4f}")
print(f"RMSE на test: {naive_rmse:.4f}")

Средний рейтинг в train: 3.7751
RMSE на test: 1.2939


### 2. Базовые модели с кросс-валидацией

In [13]:
models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(random_state=RANDOM_STATE),
    'DecisionTree': DecisionTreeRegressor(random_state=RANDOM_STATE),
    'KNeighbors': KNeighborsRegressor(),
    'RandomForest': RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
}

results = []

for name, model in models.items():
    cv_scores = cross_val_score(
        model, 
        X_train_val, 
        y_train_val, 
        cv=5, 
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    
    cv_rmse_mean = -cv_scores.mean()
    cv_rmse_std = cv_scores.std()
    
    model.fit(X_train, y_train)
    y_pred_valid = model.predict(X_valid)
    valid_rmse = root_mean_squared_error(y_valid, y_pred_valid)
    
    results.append({
        'Model': name,
        'CV RMSE (mean)': cv_rmse_mean,
        'CV RMSE (std)': cv_rmse_std,
        'Valid RMSE': valid_rmse
    })
    
    print(f"{name}:")
    print(f"CV RMSE: {cv_rmse_mean:.4f} ± {cv_rmse_std:.4f}")
    print(f"Valid RMSE: {valid_rmse:.4f}")
    print()

LinearRegression:
CV RMSE: 1.2065 ± 0.0234
Valid RMSE: 1.1971

Ridge:
CV RMSE: 1.2023 ± 0.0229
Valid RMSE: 1.1910

DecisionTree:
CV RMSE: 1.6566 ± 0.0228
Valid RMSE: 1.6344

KNeighbors:
CV RMSE: 1.3220 ± 0.0152
Valid RMSE: 1.3070

RandomForest:
CV RMSE: 1.3061 ± 0.0167
Valid RMSE: 1.3038



**Вывод**
Модели работают плохо и почти не превосходят наивный регрессор (1.2652). Модель Ridge показала лучший результат - 1.2023.

### 3. Подбор гиперпараметров для лучших моделей

In [14]:
print("\nRidge Regression")
ridge_params = {'alpha': [0.1, 1.0, 10.0, 100.0, 1000.0]}
ridge_grid = GridSearchCV(Ridge(random_state=RANDOM_STATE), ridge_params, 
                          cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
ridge_grid.fit(X_train_val, y_train_val)
print(f"Лучшие параметры: {ridge_grid.best_params_}")
print(f"Лучший CV RMSE: {-ridge_grid.best_score_:.4f}")

print("\nRandom Forest")
rf_params = {
    'n_estimators': [50, 100, 150],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}
rf_grid = GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1), 
                       rf_params, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
rf_grid.fit(X_train_val, y_train_val)
print(f"Лучшие параметры: {rf_grid.best_params_}")
print(f"Лучший CV RMSE: {-rf_grid.best_score_:.4f}")


Ridge Regression
Лучшие параметры: {'alpha': 10.0}
Лучший CV RMSE: 1.1969

Random Forest
Лучшие параметры: {'max_depth': 20, 'min_samples_split': 10, 'n_estimators': 150}
Лучший CV RMSE: 1.2076


### 4. Построение ансамблей

In [15]:
 # Модели для ансамбля
best_ridge = ridge_grid.best_estimator_
best_rf = rf_grid.best_estimator_

# 1. Voting Regressor
print("\nVoting Regressor")
voting = VotingRegressor([('ridge', best_ridge), ('rf', best_rf)])
voting_cv = -cross_val_score(voting, X_train_val, y_train_val, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1).mean()
print(f"Voting CV RMSE: {voting_cv:.4f}")

# 2. Stacking Regressor
print("\nStacking Regressor")
stacking = StackingRegressor(
    estimators=[('ridge', best_ridge), ('rf', best_rf)],
    final_estimator=LinearRegression(),
    cv=5
)
stacking_cv = -cross_val_score(stacking, X_train_val, y_train_val, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1).mean()
print(f"Stacking CV RMSE: {stacking_cv:.4f}")


Voting Regressor
Voting CV RMSE: 1.1911

Stacking Regressor
Stacking CV RMSE: 1.1909


### 5. Оценка на тестовой выборке

In [16]:
models_to_test = {
    'Ridge': ridge_grid.best_estimator_,
    'RandomForest': rf_grid.best_estimator_,
    'Voting Ensemble': voting,
    'Stacking Ensemble': stacking
}

best_test_rmse = float('inf')
best_model_name = ""
best_model_obj = None

for name, model in models_to_test.items():
    model.fit(X_train_val, y_train_val)
    y_pred = model.predict(X_test)
    rmse = root_mean_squared_error(y_test, y_pred)
    print(f"{name} Test RMSE: {rmse:.4f}")
    
    if rmse < best_test_rmse:
        best_test_rmse = rmse
        best_model_name = name
        best_model_obj = model

print(f"\nЛучшая модель: {best_model_name}")
print(f"Финальный Test RMSE: {best_test_rmse:.4f}")

joblib.dump(best_model_obj, 'data/best_regression_model.pkl')
print(f"Модель сохранена в best_regression_model.pkl")

# Сохраняем названия признаков (колонок)
# В основном скрипте нам нужно будет привести входные данные 
# к такому же виду, как при обучении
joblib.dump(list(X.columns), 'data/feature_names.pkl')
print(f"Названия признаков ({len(X.columns)} шт.) сохранены в feature_names.pkl")

Ridge Test RMSE: 1.2511
RandomForest Test RMSE: 1.2526
Voting Ensemble Test RMSE: 1.2414
Stacking Ensemble Test RMSE: 1.2421

Лучшая модель: Voting Ensemble
Финальный Test RMSE: 1.2414
Модель сохранена в best_regression_model.pkl
Названия признаков (337 шт.) сохранены в feature_names.pkl


**Вывод**
Voting Ensemble показал лучший результат RMSE - 1.2414.

## Классификация

### Базовые модели, rating - от 0 до 5, сравнение с наивным классификатором

In [17]:
def search_hyperparams(model_class, param_grid, X, y, cv, extra_params=None, scoring='accuracy'):
    extra_params = extra_params or {}
    keys = list(param_grid.keys())
    values = list(param_grid.values())
    combinations = list(itertools.product(*values))

    results = []
    for combo in tqdm(combinations, desc=model_class.__name__):
        params = dict(zip(keys, combo))
        model = model_class(**params, **extra_params)
        scores = cross_val_score(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
        results.append({
            "params": params,
            "mean_score": scores.mean(),
            "std_score": scores.std()
        })

    return pd.DataFrame(results).sort_values('mean_score', ascending=False).reset_index(drop=True)


def run_full_pipeline(X, y, model_configs, metric_func, metric_name, test_size=0.2, valid_size=0.1):
    X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y)
    X_train, X_valid, y_train, y_valid = train_test_split(X_train_val, y_train_val, test_size=valid_size, random_state=RANDOM_STATE, stratify=y_train_val)

    print(f'Train: {X_train.shape}, Valid: {X_valid.shape}, Test: {X_test.shape}')
    print(y.value_counts())

    X_combined = pd.concat([X_train, X_valid])
    y_combined = pd.concat([y_train, y_valid])
    test_fold = np.concatenate([np.full(len(X_train), -1), np.full(len(X_valid), 0)])
    ps = PredefinedSplit(test_fold)

    scorer = make_scorer(metric_func)

    all_results = {}
    for name, config in model_configs.items():
        all_results[name] = search_hyperparams(
            model_class=config['model_class'],
            param_grid=config['param_grid'],
            X=X_combined,
            y=y_combined,
            cv=ps,
            extra_params=config['extra_params'],
            scoring=scorer
        )
        print(f'{name}: best mean_score ({metric_name}) = {all_results[name].iloc[0]["mean_score"]:.4f}')

    best_scores = {name: df.iloc[0]['mean_score'] for name, df in all_results.items()}
    best_name = max(best_scores, key=best_scores.get)
    best_params = all_results[best_name].iloc[0]['params']

    print(f'\nЛучшая модель по valid: {best_name} ({best_scores[best_name]:.4f})')
    print(f'Params: {best_params}')

    best_model = model_configs[best_name]['model_class'](
        **best_params, **model_configs[best_name]['extra_params']
    )
    best_model.fit(X_train, y_train)
    test_pred = best_model.predict(X_test)
    test_score = metric_func(y_test, test_pred)

    print(f'\n{metric_name} на тесте ({best_name}): {test_score:.4f}')
    print(classification_report(y_test, test_pred))

    most_frequent = y_train.value_counts().idxmax()
    naive_pred = np.full(len(y_test), fill_value=most_frequent)
    naive_score = metric_func(y_test, naive_pred)

    print(f'\nНаивный классификатор (всегда "{most_frequent}"): {metric_name} = {naive_score:.4f}')
    print(classification_report(y_test, naive_pred))

    print(f'\n=== Итоговое сравнение ({metric_name}) ===')
    for name, score in best_scores.items():
        print(f'{name} (valid): {score:.4f}')
    print(f'Наивный классификатор (тест): {naive_score:.4f}')
    print(f'Лучшая модель ({best_name}, тест): {test_score:.4f}')

    return {
        'all_results': all_results,
        'best_model': best_model,
        'best_name': best_name,
        'best_params': best_params,
        'test_score': test_score,
        'naive_score': naive_score,
        'X_train': X_train, 'X_valid': X_valid, 'X_test': X_test,
        'y_train': y_train, 'y_valid': y_valid, 'y_test': y_test,
    }

In [18]:
model_configs = {
    'RandomForest': {
        'model_class': RandomForestClassifier,
        'param_grid': {
            'n_estimators': [100, 200, 500],
            'max_depth': [10, 20, 30, None],
            'min_samples_split': [2, 5, 10],
            'class_weight': [None, 'balanced']
        },
        'extra_params': {'n_jobs': -1, 'random_state': RANDOM_STATE}
    },
    'DecisionTree': {
        'model_class': DecisionTreeClassifier,
        'param_grid': {
            'max_depth': [5, 7, 10, 20, None],
            'min_samples_split': [2, 5, 10],
            'class_weight': [None, 'balanced']
        },
        'extra_params': {'random_state': RANDOM_STATE}
    },
    'LogisticRegression': {
        'model_class': LogisticRegression,
        'param_grid': {
            'C': [0.01, 0.1, 1, 10],
            'penalty': ['l2', None],
            'class_weight': [None, 'balanced']
        },
        'extra_params': {'max_iter': 1000, 'random_state': RANDOM_STATE}
    }
}

In [19]:
y_class = y.round().astype(int)
result_6class = run_full_pipeline(X, y_class, model_configs, metric_func=accuracy_score, metric_name='accuracy')

Train: (10121, 337), Valid: (1125, 337), Test: (2812, 337)
rating
4    9605
5    1771
0    1100
3    1043
2     431
1     108
Name: count, dtype: int64


RandomForestClassifier:   0%|          | 0/72 [00:00<?, ?it/s]

RandomForest: best mean_score (accuracy) = 0.6889


DecisionTreeClassifier:   0%|          | 0/30 [00:00<?, ?it/s]

DecisionTree: best mean_score (accuracy) = 0.6871


LogisticRegression:   0%|          | 0/16 [00:00<?, ?it/s]

LogisticRegression: best mean_score (accuracy) = 0.6880

Лучшая модель по valid: RandomForest (0.6889)
Params: {'n_estimators': 500, 'max_depth': 30, 'min_samples_split': 2, 'class_weight': None}

accuracy на тесте (RandomForest): 0.6924
              precision    recall  f1-score   support

           0       0.72      0.14      0.24       220
           1       0.00      0.00      0.00        22
           2       0.00      0.00      0.00        86
           3       0.00      0.00      0.00       209
           4       0.70      0.99      0.82      1921
           5       0.44      0.03      0.06       354

    accuracy                           0.69      2812
   macro avg       0.31      0.19      0.19      2812
weighted avg       0.59      0.69      0.58      2812


Наивный классификатор (всегда "4"): accuracy = 0.6831
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       220
           1       0.00      0.00      0.00        22
 

**Выводы**: предсказание наивного классификатора мажорным классом (rating = 4) даёт на 0.01 хуже accuracy, чем целая обученная модель RandomForest. Случайный лес повышает свою долю верных ответов за счёт часто встречающихся значений, при этом совершенно игнорирует очень маленький классы (f1-score для 1, 2 и 3 класса равны 0, что говорит о том, что половина классов не предсказывается вообще)

## Базовые модели, rating - от 0 до 2

In [20]:
def to_group(rating):
    if rating <= 1:
        return 'bad'
    elif rating <= 3:
        return 'so-so'
    else:
        return 'great'

y_group = y_class.apply(to_group)
result_3class = run_full_pipeline(X, y_group, model_configs, metric_func=accuracy_score, metric_name='accuracy')

Train: (10121, 337), Valid: (1125, 337), Test: (2812, 337)
rating
great    11376
so-so     1474
bad       1208
Name: count, dtype: int64


RandomForestClassifier:   0%|          | 0/72 [00:00<?, ?it/s]

RandomForest: best mean_score (accuracy) = 0.8151


DecisionTreeClassifier:   0%|          | 0/30 [00:00<?, ?it/s]

DecisionTree: best mean_score (accuracy) = 0.8151


LogisticRegression:   0%|          | 0/16 [00:00<?, ?it/s]

LogisticRegression: best mean_score (accuracy) = 0.8133

Лучшая модель по valid: RandomForest (0.8151)
Params: {'n_estimators': 200, 'max_depth': 30, 'min_samples_split': 5, 'class_weight': None}

accuracy на тесте (RandomForest): 0.8190
              precision    recall  f1-score   support

         bad       0.80      0.13      0.23       242
       great       0.82      1.00      0.90      2275
       so-so       0.00      0.00      0.00       295

    accuracy                           0.82      2812
   macro avg       0.54      0.38      0.38      2812
weighted avg       0.73      0.82      0.75      2812


Наивный классификатор (всегда "great"): accuracy = 0.8090
              precision    recall  f1-score   support

         bad       0.00      0.00      0.00       242
       great       0.81      1.00      0.89      2275
       so-so       0.00      0.00      0.00       295

    accuracy                           0.81      2812
   macro avg       0.27      0.33      0.30      2

**Выводы**: группировка рейтингов привела к увеличению accuracy, однако проблема того, что модель акцентирует внимание на большем классе и игнорирует редкий, всё ещё встречается: в прошлый раз рейтинг 2 и 3 давал f1 = 0, поэтому их объединение в категорию 'so-so' так же даёт f1 = 0. Необходимо в процессе обучения отслеживать метрику, которая будет оценивать классы вне зависимости от их размера

### Обучение базовых моделей с другой метрикой

Хуже всего предсказывать хороший рейтинг, когда на самом деле он плохой, поскольку это может грозить человеку как минимум разочарованием от несовпавших ожиданий, а как максимум - отравлением. Поэтому во время обучения мы будем отслеживать f2-score: метрика, которая ставит recall выше точности ровно в 2 раза

In [21]:
f2_macro = lambda y_true, y_pred: fbeta_score(y_true, y_pred, beta=2, average='macro')

result_3class_f2 = run_full_pipeline(X, y_group, model_configs, metric_func=f2_macro, metric_name='F2-macro')

Train: (10121, 337), Valid: (1125, 337), Test: (2812, 337)
rating
great    11376
so-so     1474
bad       1208
Name: count, dtype: int64


RandomForestClassifier:   0%|          | 0/72 [00:00<?, ?it/s]

RandomForest: best mean_score (F2-macro) = 0.4243


DecisionTreeClassifier:   0%|          | 0/30 [00:00<?, ?it/s]

DecisionTree: best mean_score (F2-macro) = 0.3990


LogisticRegression:   0%|          | 0/16 [00:00<?, ?it/s]

LogisticRegression: best mean_score (F2-macro) = 0.3925

Лучшая модель по valid: RandomForest (0.4243)
Params: {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 5, 'class_weight': 'balanced'}

F2-macro на тесте (RandomForest): 0.4218
              precision    recall  f1-score   support

         bad       0.21      0.43      0.28       242
       great       0.85      0.71      0.77      2275
       so-so       0.13      0.19      0.16       295

    accuracy                           0.63      2812
   macro avg       0.40      0.44      0.40      2812
weighted avg       0.72      0.63      0.67      2812


Наивный классификатор (всегда "great"): F2-macro = 0.3183
              precision    recall  f1-score   support

         bad       0.00      0.00      0.00       242
       great       0.81      1.00      0.89      2275
       so-so       0.00      0.00      0.00       295

    accuracy                           0.81      2812
   macro avg       0.27      0.33      0.3

По сравнению с наивным классификатором, обученная модель точнее на 11%, что немного лучше предыдущих экспериментов. Попробуем построить ансамбль моделей

### Ансамбли

In [22]:
f2_macro = lambda y_true, y_pred: fbeta_score(y_true, y_pred, beta=2, average='macro')

best_dt_params = result_3class_f2['all_results']['DecisionTree'].iloc[0]['params']
best_rf_params = result_3class_f2['all_results']['RandomForest'].iloc[0]['params']
best_lr_params = result_3class_f2['all_results']['LogisticRegression'].iloc[0]['params']

X_train = result_3class_f2['X_train']
X_valid = result_3class_f2['X_valid']
X_test = result_3class_f2['X_test']
y_train = result_3class_f2['y_train']
y_valid = result_3class_f2['y_valid']
y_test = result_3class_f2['y_test']

In [23]:
dt_model = DecisionTreeClassifier(**best_dt_params, random_state=RANDOM_STATE)
rf_model = RandomForestClassifier(**best_rf_params, random_state=RANDOM_STATE, n_jobs=-1)
lr_model = LogisticRegression(**best_lr_params, random_state=RANDOM_STATE)

weight_options = [[1, 1], [1, 2], [2, 1], [1, 3], [3, 1], [1, 4]]

voting_results = []
for weights in tqdm(weight_options):
    voting_model = VotingClassifier(
        estimators=[('dt', dt_model), ('rf', rf_model)],
        voting='soft',
        weights=weights
    )
    voting_model.fit(X_train, y_train)
    pred = voting_model.predict(X_valid)
    score = f2_macro(y_valid, pred)
    voting_results.append({'weights': weights, 'f2_macro': score})

voting_df = pd.DataFrame(voting_results).sort_values('f2_macro', ascending=False)
print(voting_df)

best_voting_weights = voting_df.iloc[0]['weights']
print(f'\nЛучшие веса Voting: {best_voting_weights}, F2-macro (valid): {voting_df.iloc[0]["f2_macro"]:.4f}')

  0%|          | 0/6 [00:00<?, ?it/s]

  weights  f2_macro
5  [1, 4]  0.418683
3  [1, 3]  0.412477
1  [1, 2]  0.397407
0  [1, 1]  0.386988
2  [2, 1]  0.386517
4  [3, 1]  0.385738

Лучшие веса Voting: [1, 4], F2-macro (valid): 0.4187


In [24]:
stacking_model = StackingClassifier(
    estimators=[
        ('dt', DecisionTreeClassifier(**best_dt_params, random_state=RANDOM_STATE)),
        ('rf', RandomForestClassifier(**best_rf_params, random_state=RANDOM_STATE, n_jobs=-1)),
        ('lr', LogisticRegression(**best_lr_params, random_state=RANDOM_STATE))
    ],
    final_estimator=LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    cv=5,
    n_jobs=-1
)

stacking_model.fit(X_train, y_train)
stacking_pred_valid = stacking_model.predict(X_valid)
stacking_score_valid = f2_macro(y_valid, stacking_pred_valid)
print(f'Stacking F2-macro (valid): {stacking_score_valid:.4f}')

Stacking F2-macro (valid): 0.3942


In [25]:
ensemble_scores = {
    'Voting': voting_df.iloc[0]['f2_macro'],
    'Stacking': stacking_score_valid
}

best_ensemble_name = max(ensemble_scores, key=ensemble_scores.get)
print(f'\nЛучший ансамбль по valid: {best_ensemble_name} ({ensemble_scores[best_ensemble_name]:.4f})')

if best_ensemble_name == 'Voting':
    final_model = VotingClassifier(
        estimators=[('dt', dt_model), ('rf', rf_model)],
        voting='soft',
        weights=best_voting_weights
    )
    final_model.fit(X_train, y_train)
    test_pred_ens = final_model.predict(X_test)
else:
    test_pred_ens = stacking_model.predict(X_test)

test_score_ens = f2_macro(y_test, test_pred_ens)
print(f'\nF2-macro на тесте ({best_ensemble_name}): {test_score_ens:.4f}')
print(classification_report(y_test, test_pred_ens))


Лучший ансамбль по valid: Voting (0.4187)

F2-macro на тесте (Voting): 0.4166
              precision    recall  f1-score   support

         bad       0.22      0.36      0.27       242
       great       0.84      0.79      0.82      2275
       so-so       0.14      0.13      0.13       295

    accuracy                           0.69      2812
   macro avg       0.40      0.43      0.41      2812
weighted avg       0.71      0.69      0.70      2812



In [26]:
print('===Итоговое сравнение (F2-macro на тесте)===')
print(f'Наивный классификатор: {result_3class_f2["naive_score"]:.4f}')
print(f'Лучшая одиночная модель (RandomForest): {result_3class_f2["test_score"]:.4f}')
print(f'Лучший ансамбль ({best_ensemble_name}): {test_score_ens:.4f}')

===Итоговое сравнение (F2-macro на тесте)===
Наивный классификатор: 0.3183
Лучшая одиночная модель (RandomForest): 0.4218
Лучший ансамбль (Voting): 0.4166


Чтобы сделать окончательные выводы, посмотрим топ-корреляций между rating и ингредиентами

In [27]:
y_numeric = y_group.map({'bad': 0, 'so-so': 1, 'great': 2})
correlations = X_train.corrwith(y_numeric.loc[X_train.index]).abs().sort_values(ascending=False)
print(correlations.head(10))

gin               0.197801
bitters           0.133989
rum               0.090355
brandy            0.078812
chartreuse        0.074219
créme de cacao    0.068259
vermouth          0.067965
fruit juice       0.062086
lime juice        0.061866
chile pepper      0.060165
dtype: float64


**Выводы**: Ансамблирование (Voting) работает хуже одиночной модели RandomForest. Это подтверждает, что основное ограничение качества классификации связано не с выбором алгоритма, а с информативностью признаков: корреляция отдельных ингредиентов с рейтингом невелика (макс. 0.19), и рейтинг, по-видимому, определяется более сложными факторами (пропорции, техника приготовления, сочетания ингредиентов), которые не отражены в текущем OHE-представлении. Дальнейшее повышение качества потребовало бы расширения набора признаков, а не более сложных моделей или ансамблей.

Лучшая модель классификации - случайный лес

In [28]:
best_model = result_3class_f2['best_model']
joblib.dump(best_model, 'data/best_classifier.pkl')

['data/best_classifier.pkl']

## Решение

Чтобы сравнить регрессор и классификатор объективно, необходимо рассматривать одинаковые метрики качества. Для этого мы возьмём предсказания регрессора, переведём их в категориальную шкалу и сравним метрику f2

In [29]:
X_train_val_aligned = pd.concat([X_train, X_valid])
y_train_val_aligned = y.loc[X_train_val_aligned.index]
y_test_raw = y.loc[X_test.index]

voting.fit(X_train_val_aligned, y_train_val_aligned)
y_pred_reg = voting.predict(X_test)

y_pred_class_from_reg = np.array([to_group(round(r)) for r in y_pred_reg])
labels_order = ['bad', 'so-so', 'great']

f2_macro_from_regression = fbeta_score(
    y_test,
    y_pred_class_from_reg,
    beta=2,
    average='macro',
    labels=labels_order
)

print(f"F2-macro (регрессия Voting -> округление -> группы): {f2_macro_from_regression:.4f}")
print(f"F2-macro (случайный лес): {result_3class_f2["test_score"]:.4f}")

print("\n===Метрики регрессии, переведённой в группы===")
print(classification_report(y_test, y_pred_class_from_reg, labels=labels_order))

F2-macro (регрессия Voting -> округление -> группы): 0.3588
F2-macro (случайный лес): 0.4218

===Метрики регрессии, переведённой в группы===
              precision    recall  f1-score   support

         bad       0.86      0.07      0.14       242
       so-so       0.09      0.05      0.07       295
       great       0.83      0.95      0.89      2275

    accuracy                           0.78      2812
   macro avg       0.59      0.36      0.36      2812
weighted avg       0.75      0.78      0.74      2812



Лучшим решением будет взять **классификатор**, поскольку он на 14% лучше определяет класс 'bad' и на 9% лучше определаяет класс 'so-so', при этом на 12% хуже определяет 'great'

## Пищевая ценность

In [30]:
# 1. API ключ
API_KEY = "M9xvs7fdKf6x780T1QoaNiku8UbRZSK0K6HgSRRv"

# 2. Загружаем список ингредиентов
feature_names = joblib.load('data/feature_names.pkl')
print(f"Загружено {len(feature_names)} ингредиентов для обработки.")

# 3. Словарь суточных норм
daily_values = {
    # DRVs (Daily Reference Values)
    'Total Fat': 78,            # g
    'Saturated Fat': 20,        # g
    'Cholesterol': 300,         # mg
    'Total Carbohydrate': 275,  # g
    'Dietary Fiber': 28,        # g
    'Sodium': 2300,             # mg
    'Protein': 50,              # g
    'Added Sugars': 50,         # g
    
    # RDIs (Reference Daily Intakes)
    'Vitamin D': 20,            # mcg
    'Vitamin A': 900,           # mcg RAE
    'Vitamin C': 90,            # mg
    'Calcium': 1300,            # mg
    'Iron': 18,                 # mg
    'Vitamin E': 15,            # mg
    'Vitamin K': 120,           # mcg
    'Thiamin': 1.2,             # mg
    'Riboflavin': 1.3,          # mg
    'Niacin': 16,               # mg
    'Vitamin B6': 1.7,          # mg
    'Folate': 400,              # mcg DFE
    'Vitamin B12': 2.4,         # mcg
    'Biotin': 30,               # mcg
    'Pantothenic acid': 5,      # mg
    'Phosphorus': 1250,         # mg
    'Iodine': 150,              # mcg
    'Magnesium': 420,           # mg
    'Zinc': 11,                 # mg
    'Selenium': 55,             # mcg
    'Copper': 0.9,              # mg
    'Manganese': 2.3,           # mg
    'Chromium': 35,             # mcg
    'Molybdenum': 45,           # mcg
    'Chloride': 2300,           # mg
    'Potassium': 4700,          # mg
    'Choline': 550              # mg
}

# Словарь альтернативных названий для некоторых ингредиентов
ingredient_alternatives = {
    'green onion/scallion': 'green onion',
    'milk/cream': 'milk',
    'sweet potato/yam': 'sweet potato',
    'breadcrumbs': 'bread crumbs',
    'marscarpone': 'mascarpone',
    'phyllo/puff pastry dough': 'phyllo',
    'jelly': 'jam or jelly',
    'jam': 'jam or jelly',
}

# 4. Функция запроса к API
def get_nutrition_usda(ingredient):
    """
    Ищет продукт в базе USDA FoodData Central и возвращает нутриенты на 100г.
    Выбирает вариант с наибольшим количеством заполненных нутриентов.
    """
    url = f"https://api.nal.usda.gov/fdc/v1/foods/search?query={ingredient}&api_key={API_KEY}&pageSize=5"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if 'foods' in data and len(data['foods']) > 0:
                
                # Выбираем продукт с наибольшим количеством нутриентов
                best_food = None
                max_nutrients = 0
                
                for food in data['foods']:
                    num_nutrients = len(food.get('foodNutrients', []))
                    if num_nutrients > max_nutrients:
                        max_nutrients = num_nutrients
                        best_food = food
                
                # Если нашли продукт с достаточным количеством данных (больше 3)
                if best_food and max_nutrients > 3:
                    # ИСПРАВЛЕНИЕ: используем 'nutrientName' и 'value' вместо 'nutrient' и 'amount'
                    nutriments = {n['nutrientName']: n.get('value', 0) 
                                 for n in best_food.get('foodNutrients', [])}
                    
                    # Маппинг названий из API на наши ключи
                    mapping = {
                        'Protein': 'Protein',
                        'Total lipid (fat)': 'Total Fat',
                        'Saturated fatty acids': 'Saturated Fat',
                        'Cholesterol': 'Cholesterol',
                        'Carbohydrate, by difference': 'Total Carbohydrate',
                        'Fiber, total dietary': 'Dietary Fiber',
                        'Sugars, added': 'Added Sugars',
                        'Sodium, Na': 'Sodium',
                        'Potassium, K': 'Potassium',
                        'Calcium, Ca': 'Calcium',
                        'Iron, Fe': 'Iron',
                        'Vitamin D (D2 + D3)': 'Vitamin D',
                        'Vitamin A, RAE': 'Vitamin A',
                        'Vitamin C, total ascorbic acid': 'Vitamin C',
                        'Vitamin E (alpha-tocopherol)': 'Vitamin E',
                        'Vitamin K (phylloquinone)': 'Vitamin K',
                        'Thiamin': 'Thiamin',
                        'Riboflavin': 'Riboflavin',
                        'Niacin': 'Niacin',
                        'Vitamin B-6': 'Vitamin B6',
                        'Folate, total': 'Folate',
                        'Vitamin B-12': 'Vitamin B12',
                        'Biotin': 'Biotin',
                        'Pantothenic acid': 'Pantothenic acid',
                        'Phosphorus, P': 'Phosphorus',
                        'Iodine, I': 'Iodine',
                        'Magnesium, Mg': 'Magnesium',
                        'Zinc, Zn': 'Zinc',
                        'Selenium, Se': 'Selenium',
                        'Copper, Cu': 'Copper',
                        'Manganese, Mn': 'Manganese',
                        'Chromium, Cr': 'Chromium',
                        'Molybdenum, Mo': 'Molybdenum',
                        'Chloride, Cl': 'Chloride',
                        'Choline, total': 'Choline'
                    }
                    
                    result = {}
                    for api_name, our_name in mapping.items():
                        val = nutriments.get(api_name, 0)
                        if val > 0:
                            result[our_name] = val
                    return result
        return {}
    except Exception:
        return {}

def get_nutrition_with_fallback(ingredient):
    """Пробует оригинальное название, а если не находит — альтернативное"""
    # 1. Пробуем оригинальное название
    result = get_nutrition_usda(ingredient)
    
    # 2. Если не нашли (меньше 4 нутриентов) и есть альтернатива
    if len(result) < 4 and ingredient in ingredient_alternatives:
        alt_name = ingredient_alternatives[ingredient]
        alt_result = get_nutrition_usda(alt_name)
        if len(alt_result) > len(result):
            return alt_result
            
    return result

# 5. Основной цикл с прогресс-баром
print("\n" + "="*60)
print("СБОР ДАННЫХ ИЗ USDA API")
print("="*60)

results = []
found_count = 0

for ing in tqdm(feature_names, desc="Обработка ингредиентов", unit="ингр"):
    nutrients = get_nutrition_with_fallback(ing)
    
    if len(nutrients) >= 3:  # проверяем, что найдено хотя бы 3 нутриента
        found_count += 1
        pct_dv = {}
        for nutrient, dv_amount in daily_values.items():
            amount = nutrients.get(nutrient, 0)
            if amount > 0:
                # Формула: (количество в продукте на 100г / суточная норма) × 100
                pct = (amount / dv_amount) * 100
                pct_dv[f"{nutrient} (% DV)"] = round(pct, 1)
            else:
                pct_dv[f"{nutrient} (% DV)"] = 0.0
        
        pct_dv['Ingredient'] = ing
        results.append(pct_dv)
    
    time.sleep(0.6)  # Пауза, чтобы не превысить лимит API (120 запросов/мин)

# 6. Сохранение результатов
print("\n" + "="*60)
print("СОХРАНЕНИЕ РЕЗУЛЬТАТОВ")
print("="*60)

if results:
    nutrition_df = pd.DataFrame(results)
    nutrition_df.set_index('Ingredient', inplace=True)
    nutrition_df.to_csv('data/nutrition_facts.csv')
    print(f"Файл data/nutrition_facts.csv успешно сохранен!")
    print(f"Найдено в API: {found_count} из {len(feature_names)}")
    
    print("\nПример данных (первые 3 ингредиента):")
    display(nutrition_df.head(3))
else:
    print("Не удалось получить данные ни для одного ингредиента")

Загружено 337 ингредиентов для обработки.

СБОР ДАННЫХ ИЗ USDA API


Обработка ингредиентов:   0%|          | 0/337 [00:00<?, ?ингр/s]


СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
Не удалось получить данные ни для одного ингредиента


In [31]:
# Загружаем исходный список всех ингредиентов
feature_names = joblib.load('data/feature_names.pkl')

# Загружаем сохраненный DataFrame с найденными данными
nutrition_df = pd.read_csv('data/nutrition_facts.csv', index_col='Ingredient')

# Находим разницу
found_ingredients = set(nutrition_df.index.str.lower())
all_ingredients = set([ing.lower() for ing in feature_names])

not_found = all_ingredients - found_ingredients

print("="*60)
print(f"НЕ НАЙДЕНЫ В API: {len(not_found)} ингредиентов")
print("="*60)

for i, ing in enumerate(sorted(not_found), 1):
    print(f"{i:2d}. {ing}")

print(f"\nВсего найдено: {len(found_ingredients)}")
print(f"Всего не найдено: {len(not_found)}")
print(f"Всего было: {len(all_ingredients)}")

НЕ НАЙДЕНЫ В API: 20 ингредиентов
 1. basil
 2. butterscotch/caramel
 3. campari
 4. cilantro
 5. cinnamon
 6. cognac/armagnac
 7. grappa
 8. jícama
 9. kahlúa
10. lingonberry
11. midori
12. nutmeg
13. oregano
14. paprika
15. parsley
16. peanut
17. rosemary
18. rosé
19. sherry
20. thyme

Всего найдено: 317
Всего не найдено: 20
Всего было: 337


## Похожие рецепты

In [12]:
OUTPUT_CSV = "data/links.csv"
CHECKPOINT_EVERY = 25
CONCURRENCY = 2
MAX_PAGES = 40
DISCOVERED_API_LOG = "data/links.log"


def log_api_call(url, body_snippet):
    with open(DISCOVERED_API_LOG, "a", encoding="utf-8") as f:
        f.write(f"{url}\n{body_snippet[:500]}\n{'-'*80}\n")


def normalize(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def slug_words_from_url(url: str) -> set:
    """
    Достаёт последний сегмент URL (slug) и превращает его в множество слов.
    Пример: .../korean-marinated-beef-109206 -> {'korean','marinated','beef','109206'}
    """
    path = url.split("?")[0].rstrip("/")
    slug = path.split("/")[-1].lower()
    slug = re.sub(r"[^\w\s-]", "", slug)
    words = {w for w in slug.split("-") if w}
    return words


def find_exact(title_norm: str, candidates):
    """
    1) Сначала ищем точное совпадение нормализованного текста ссылки с title.
    2) Если точного нет — ищем рецепт, в slug которого встречаются ВСЕ слова
       из title (порядок и лишние слова в slug не важны).
    Возвращает (link, match_type) или (None, None).
    """
    title_words = set(title_norm.split())

    for text, link in candidates:
        if normalize(text) == title_norm:
            return link, "exact_title"

    for text, link in candidates:
        slug_words = slug_words_from_url(link)
        if title_words and title_words.issubset(slug_words):
            return link, "all_words_in_slug"

    return None, None


async def dismiss_cookie_banner(page):
    selectors = [
        "button:has-text('Accept All')",
        "button:has-text('Accept all')",
        "#fides-banner-button-accept",
        "[id*='accept']",
        ".fides-banner-button-primary",
    ]
    for sel in selectors:
        try:
            btn = page.locator(sel)
            if await btn.count() > 0:
                await btn.first.click(timeout=3000)
                await page.wait_for_timeout(500)
                return True
        except Exception:
            continue
    try:
        await page.evaluate("""
            () => {
                const overlay = document.querySelector('#fides-overlay');
                if (overlay) overlay.remove();
            }
        """)
    except Exception:
        pass
    return False


async def get_current_links(page):
    links = await page.eval_on_selector_all(
        "a[href*='/recipes/']",
        "els => els.map(e => ({text: e.innerText, href: e.href}))",
    )
    return [(l["text"], l["href"]) for l in links if l["text"]]


async def search_one(context, title: str):
    page = await context.new_page()
    title_norm = normalize(title)
    seen_set = set()
    seen_candidates = []
    debug = {"pages_visited": 0, "stopped_reason": "", "match_type": None}

    async def on_response(response):
        url = response.url
        if "json" in (response.headers.get("content-type", "")) and (
            "search" in url.lower() or "recipe" in url.lower() or "algolia" in url.lower()
        ):
            try:
                body = await response.text()
                log_api_call(url, body)
            except Exception:
                pass

    page.on("response", on_response)

    def add_candidates(new_links):
        added = 0
        for item in new_links:
            if item not in seen_set:
                seen_set.add(item)
                seen_candidates.append(item)
                added += 1
        return added

    try:
        query = quote(title)

        await page.goto(
            f"https://www.epicurious.com/search?q={query}",
            wait_until="networkidle",
            timeout=20000,
        )
        await page.wait_for_selector("a[href*='/recipes/']", timeout=8000)
        await dismiss_cookie_banner(page)

        links = await get_current_links(page)
        add_candidates(links)

        link, match_type = find_exact(title_norm, seen_candidates)
        if link:
            debug["stopped_reason"] = f"found_on_page_1_{match_type}"
            debug["match_type"] = match_type
            await page.close()
            return link, seen_candidates, debug

        for page_num in range(2, MAX_PAGES + 1):
            try:
                await page.goto(
                    f"https://www.epicurious.com/search?q={query}&page={page_num}",
                    wait_until="networkidle",
                    timeout=15000,
                )
                await dismiss_cookie_banner(page)
                debug["pages_visited"] += 1
            except Exception:
                debug["stopped_reason"] = "goto_failed"
                break

            links = await get_current_links(page)
            new_count = add_candidates(links)

            if new_count == 0:
                debug["stopped_reason"] = "no_new_candidates_stopped_pagination"
                break

            link, match_type = find_exact(title_norm, seen_candidates)
            if link:
                debug["stopped_reason"] = f"found_on_page_{page_num}_{match_type}"
                debug["match_type"] = match_type
                await page.close()
                return link, seen_candidates, debug

        if not debug["stopped_reason"]:
            debug["stopped_reason"] = "max_pages_reached"

    except Exception as e:
        seen_candidates.append(("ERROR", str(e)))
        debug["stopped_reason"] = "exception"

    await page.close()
    return None, seen_candidates, debug


async def worker(name, queue, context, results, done_titles):
    while not queue.empty():
        i, title = await queue.get()
        if title in done_titles:
            queue.task_done()
            continue

        link, seen_candidates, debug = await search_one(context, title)

        results.append({
            "title": title,
            "epicurious_url": link,
            "found_exact": link is not None,
            "match_type": debug.get("match_type"),
            "n_candidates_seen": len(seen_candidates),
            "pages_visited": debug["pages_visited"],
            "stopped_reason": debug["stopped_reason"],
        })

        if len(results) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
            print(f"[{name}] сохранено {len(results)} строк...")

        await asyncio.sleep(random.uniform(1.5, 3.5))
        queue.task_done()


async def run_search(df, title_col="title"):
    try:
        done_df = pd.read_csv(OUTPUT_CSV)
        done_titles = set(done_df["title"])
        results = done_df.to_dict("records")
    except FileNotFoundError:
        done_titles = set()
        results = []

    queue = asyncio.Queue()
    for i, row in df.iterrows():
        queue.put_nowait((i, row[title_col]))

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
            )
        )

        workers = [
            asyncio.create_task(worker(f"w{i}", queue, context, results, done_titles))
            for i in range(CONCURRENCY)
        ]
        await queue.join()
        for w in workers:
            w.cancel()

        await browser.close()

    pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
    return pd.DataFrame(results)

In [13]:
BATCH_SIZE = 1000
PAUSE_BETWEEN_BATCHES = 180

for start in range(0, len(df), BATCH_SIZE):
    batch = df.iloc[start:start + BATCH_SIZE]
    print(f"=== Батч {start}-{start + len(batch)} ===")

    result_df = await run_search(batch, title_col="title")

    print(f"Батч завершён, всего в CSV: {len(result_df)} строк")

    if start + BATCH_SIZE < len(df):
        print(f"Отдых {PAUSE_BETWEEN_BATCHES} сек...")
        await asyncio.sleep(PAUSE_BETWEEN_BATCHES)

print("Готово!")

=== Батч 0-1000 ===
Батч завершён, всего в CSV: 1500 строк
Отдых 180 сек...
=== Батч 1000-2000 ===
[w1] сохранено 1525 строк...
[w0] сохранено 1550 строк...
[w0] сохранено 1575 строк...
[w1] сохранено 1600 строк...
[w1] сохранено 1625 строк...
[w0] сохранено 1650 строк...
[w0] сохранено 1675 строк...
[w1] сохранено 1700 строк...
[w1] сохранено 1725 строк...
[w1] сохранено 1750 строк...
[w1] сохранено 1775 строк...
[w1] сохранено 1800 строк...
[w0] сохранено 1825 строк...
[w0] сохранено 1850 строк...
[w0] сохранено 1875 строк...
[w1] сохранено 1900 строк...
[w1] сохранено 1925 строк...
[w0] сохранено 1950 строк...
[w1] сохранено 1975 строк...
Батч завершён, всего в CSV: 1998 строк
Отдых 180 сек...
=== Батч 2000-3000 ===
[w1] сохранено 2000 строк...
[w1] сохранено 2025 строк...
[w1] сохранено 2050 строк...
[w1] сохранено 2075 строк...
[w1] сохранено 2100 строк...
[w0] сохранено 2125 строк...
[w0] сохранено 2150 строк...
[w1] сохранено 2175 строк...
[w1] сохранено 2200 строк...
[w1] сохра

Поскольку не сразу было замечено, что не все дубликаты удалены (а именно не удалились дубликаты, у которых отличался только рейтинг, потому что рейтинг - величина не константная), из полученных url-адресов их тоже нужно удалить

In [ ]:
links_df = pd.read_csv('data/links.csv')
links_dup = links_df[links_df['title'].duplicated(keep=False)].sort_values('title')
links_dup[['title', 'epicurious_url']]

In [ ]:
links_df = links_df.drop_duplicates(subset="title", keep="first")
print(f'Количество строк обработанного датасета и датасета ссылок совпадают: {df.shape[0] == links_df.shape[0]}')

Рецепты, для которых не удалось найти ссылки, либо удалены, либо их url очень сильно не совпадает с названием рецепта. В таком случае оставим ссылку на главную страницу

In [ ]:
print(f'Сколько рецептов остались без ссылки: {links_df['epicurious_url'].isnull().sum()}')
links_df['epicurious_url'] = links_df['epicurious_url'].fillna('https://www.epicurious.com')
print(f'После обработки: {links_df['epicurious_url'].isnull().sum()}')

In [ ]:
recipes_df = pd.read_json('data/full_format_recipes.json')
recipes_df = recipes_df.drop_duplicates(subset='title', keep='last')
recipes_df.head()

In [ ]:
final_df = links_df[['title', 'epicurious_url']].merge(
    recipes_df,
    on='title',
    how='left'
)

print(f'Размерность итогового файла: {final_df.shape}')

In [ ]:
final_df.to_csv('data/recipes_info.csv', index=False)
print('Успешно сохранено!')